In [ ]:
from models import ModelMgr
import kagglehub
import ast
import pandas as pd

from models.semantic_validation import LLaMAValidationModel

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

from tqdm import tqdm
tqdm.pandas()

In [ ]:
lvm = LLaMAValidationModel(ModelMgr())
lvm

In [ ]:
# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

print("Path to dataset files:", path)

In [ ]:
# df_ratings_small = pd.read_csv(f"{path}/ratings_small.csv")
# df_links = pd.read_csv(f"{path}/links.csv")
# df_links_small = pd.read_csv(f"{path}/links_small.csv")
# df_credits = pd.read_csv(f"{path}/credits.csv")
# df_ratings = pd.read_csv(f"{path}/ratings.csv")

df_movies_metadata = pd.read_csv(f"{path}/movies_metadata.csv", low_memory=False, index_col="id").drop(index='1997-08-20').drop(index="2012-09-29")

df_keywords = pd.read_csv(f"{path}/keywords.csv", index_col="id")
df_keywords.index = df_keywords.index.astype(str)

In [ ]:
df_keywords_map = df_keywords.copy()
df_keywords_map["keywords"] = df_keywords_map["keywords"].apply(lambda x: {y["name"] for y in ast.literal_eval(x)})
df_keywords_map

In [ ]:
def assign_random_keywords(current_keywords: set, df_keywords_pool: pd.DataFrame):
    selected_keywords = df_keywords_pool.sample(1)["keywords"].values[0]
    while not selected_keywords.isdisjoint(current_keywords):
        selected_keywords: set = df_keywords_pool.sample(1)["keywords"].values[0]
    return selected_keywords

In [ ]:
df_true, df_false = train_test_split(df_keywords_map, test_size=0.5)

df_true["class"] = True

df_false["keywords"] = df_false["keywords"].apply(lambda x: assign_random_keywords(x, df_true))
df_false["class"] = False

In [ ]:
df_full = pd.concat([df_true, df_false])\
    .merge(df_movies_metadata, left_index=True, right_index=True)\
    [["keywords", "overview", "original_title", "class"]]

df_full = df_full[df_full["keywords"].apply(lambda x: len(x) > 0)].sample(frac=1)

df_full.head()

In [ ]:
def eval(df, prompt_creation):
    df["prompt"] = df.apply(prompt_creation, axis=1)
    df["prediction"] = df["prompt"].progress_apply(lambda x: lvm(x))
    print(classification_report(df["class"], df["prediction"]))
    return f1_score(df["class"], df["prediction"])

In [ ]:
score = eval(df_full.head(200).copy(), lambda x: f"Do these keywords: {', '.join(x["keywords"])} describe this movie: \"{x["original_title"]}\"?")
print(score)

In [ ]:
score = eval(df_full.head(200).copy(), lambda x: f"Do these: {', '.join(x["keywords"])} describe this: \"{x["original_title"]}\"?")
print(score)

In [ ]:
score = eval(df_full.head(200).copy(), lambda x: f"Do these keywords: {', '.join(x["keywords"])} describe a movie with this description: \"{x["overview"]}\"?")
print(score)

In [ ]:
score = eval(df_full.head(200).copy(), lambda x: f"Do these: {', '.join(x["keywords"])} describe this: \"{x["overview"]}\"?")
print(score)